# EDA и baseline-модель

Исследование исходных данных и baseline CatBoost. Логика расчётов, пороги фильтрации, списки удаляемых признаков и параметры моделей сохранены. Production-обучение выполняется отдельно в [`src/train.py`](../src/train.py).

## 1. Импорты и конфигурация

Подключение библиотек и настройка путей к данным.

In [1]:
import catboost as cb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn as sk

## 2. Загрузка данных

Загружаются train- и test-наборы, после чего категориальные и числовые признаки определяются по типам test-набора.

## Loading data

In [2]:
BASE_DATA_PATH = "../data/"
response_train_path = BASE_DATA_PATH + "response_train.csv"
response_test_path = BASE_DATA_PATH + "response_test.csv"


In [3]:
raw_train_df = pd.read_csv(response_train_path)
raw_test_df = pd.read_csv(response_test_path)

In [4]:
cat_columns = raw_test_df.select_dtypes(exclude=['int64', 'float64']).columns.to_list()
num_columns = raw_test_df.select_dtypes(include=['int64', 'float64']).columns.to_list()


In [5]:
# Контроль схемы после загрузки
assert "target" in raw_train_df.columns, "В train отсутствует target"
assert set(raw_train_df.columns) - {"target"} >= set(raw_test_df.columns), (
    "В train не хватает колонок, присутствующих в test"
)
print(f"Train: {raw_train_df.shape}; test: {raw_test_df.shape}")
print(f"Категориальных признаков: {len(cat_columns)}")
print(f"Числовых признаков: {len(num_columns)}")

Train: (25101, 52); test: (5032, 51)
Категориальных признаков: 13
Числовых признаков: 38


## 3. Анализ пропусков

Сначала выполняется исходное приведение категориальных признаков к нижнему регистру, затем строится таблица долей пропусков.

#### Выявление доли пропусков

In [6]:
for col in cat_columns:
    raw_train_df[col] = raw_train_df[col].astype(str).str.lower()

In [7]:
# Создаем DataFrame со статистикой по пропускам
nan_stats = pd.DataFrame({
    'null_percent': raw_train_df.isna().mean()[raw_train_df.isna().sum() > 0].sort_values(ascending=False),
    'dtype': raw_train_df.dtypes[raw_train_df.isna().sum() > 0]
})

nan_stats

,null_percent,dtype
previous-cards,0.980598,float64
work-time,0.090116,float64


## 4. Предварительное удаление идентификаторов

Удаляются `previous-cards` и `client_id` перед дальнейшим EDA, как в исходном notebook.

In [8]:
raw_train_df.drop(columns=["previous-cards", "client_id"], inplace=True)

In [9]:
print(f"Размер после удаления идентификаторов: {raw_train_df.shape}")

Размер после удаления идентификаторов: (25101, 50)


## 5. EDA категориальных признаков

Для каждой категории оценивается доля положительного target, размер группы и стандартная ошибка. Сохраняются исходные фильтры достоверности.

# EDA

### Категориальные

Вычисление долей заинтересованности по категориям по каждому из признаков.

In [10]:
def analyze_categorical_features(df, categorical_columns, target_column):
    """Сохраняет исходные фильтры EDA для одиночных категорий."""
    tables = []
    global_mean = df[target_column].mean()

    for col in categorical_columns:
        summary_df = df.groupby(col)[target_column].agg(
            Доля_целевых="mean",
            Всего="count",
            Ошибки_доли="sem",
        ).query("Ошибки_доли < 0.05 & Всего > 30").sort_values(
            by="Доля_целевых", ascending=False
        )
        summary_df["Нижняя_граница"] = (
            summary_df["Доля_целевых"] - 1.96 * summary_df["Ошибки_доли"]
        )
        summary_df["Верхняя_граница"] = (
            summary_df["Доля_целевых"] + 1.96 * summary_df["Ошибки_доли"]
        )
        summary_df["Достоверность"] = summary_df["Нижняя_граница"] > global_mean
        tables.append(summary_df[summary_df["Достоверность"]])

    final_df = pd.concat(tables, keys=categorical_columns)
    final_df.index.names = ["Признак", "Категория"]
    return final_df.sort_values(
        by=["Признак", "Доля_целевых"], ascending=[True, False]
    )


global_mean = raw_train_df["target"].mean()

final_df = analyze_categorical_features(raw_train_df, cat_columns, "target")

final_df.groupby(["Признак"]).agg(
    Среднее=("Доля_целевых", "mean"),
    Дисперсия=("Доля_целевых", "std"),
    Максимум=("Доля_целевых", "max"),
    Минимум=("Доля_целевых", "min"),
    Всего=("Всего", "sum"),
).sort_values("Максимум", ascending=False).dropna()


,Среднее,Дисперсия,Максимум,Минимум,Всего
Признак,,,,,
postal-region,0.203861,0.061043,0.384058,0.154830,3427
fact-region,0.200572,0.050027,0.341317,0.154830,3470
region,0.180423,0.068962,0.283058,0.139254,6423
last-loan-region,0.186229,0.032629,0.272912,0.155425,3874
registration-region,0.182747,0.018335,0.216117,0.155271,3911
family-income,0.175496,0.052266,0.212454,0.138539,10809
industry,0.164321,0.013889,0.179739,0.152788,5879
tp-state,0.154916,0.017672,0.167412,0.142420,12350
title,0.151419,0.010901,0.167139,0.139127,8492


## 6. EDA пар категориальных признаков

Перебираются все пары категориальных признаков и выбираются сочетания с повышенной долей target.

In [11]:
from itertools import combinations


def find_strong_categorical_pairs(df, categorical_columns, target_column, global_mean):
    """Сохраняет исходные фильтры EDA для пар категориальных признаков."""
    results = []

    for col1, col2 in combinations(categorical_columns, 2):
        cross = pd.crosstab(
            df[col1],
            df[col2],
            values=df[target_column],
            aggfunc=["count", "mean", "sem"],
        ).round(4)
        flat = cross.stack().reset_index()
        flat.columns = [col1, col2, "count", "mean", "sem"]
        flat = flat[
            (flat["count"] >= 100)
            & (flat["sem"] < 0.05)
            & (flat["mean"] - 1.96 * flat["sem"] > global_mean)
        ].copy()
        if not flat.empty:
            flat["Признак1"] = col1
            flat["Признак2"] = col2
            flat["Комбинация"] = (
                flat[col1].astype(str) + " * " + flat[col2].astype(str)
            )
            results.append(flat[["Признак1", "Признак2", "Комбинация", "count", "mean", "sem"]])

    if not results:
        return pd.DataFrame(
            columns=["Признак1", "Признак2", "Комбинация", "count", "mean", "sem"]
        )
    return pd.concat(results, ignore_index=True).sort_values("mean", ascending=False)


final_pairs = find_strong_categorical_pairs(
    raw_train_df, cat_columns, "target", global_mean
)
final_pairs.head(20)


/tmp/ipykernel_825982/2199722595.py:15: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  flat = cross.stack().reset_index()
/tmp/ipykernel_825982/2199722595.py:15: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  flat = cross.stack().reset_index()
/tmp/ipykernel_825982/2199722595.py:15: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  flat = cross.stack().reset_index()
/

,Признак1,Признак2,Комбинация,count,mean,sem
392,job-type,postal-region,участие в основ. деятельности * москва,110.0,0.4182,0.0472
321,tp-foreign,postal-region,без участия * москва,124.0,0.4032,0.0442
552,postal-region,last-loan-region,москва * nan,124.0,0.3871,0.0439
517,fact-region,postal-region,москва * москва,135.0,0.3852,0.0420
563,postal-region,region,москва * центральный офис,131.0,0.3817,0.0426
376,job-type,fact-region,участие в основ. деятельности * москва,135.0,0.3778,0.0419
528,fact-region,last-loan-region,москва * nan,140.0,0.3571,0.0406
304,tp-foreign,fact-region,без участия * москва,152.0,0.3553,0.0389
239,tp-state,fact-region,частная компания * москва,115.0,0.3478,0.0446
540,fact-region,region,москва * центральный офис,152.0,0.3421,0.0386


## 7. Удаление региональных признаков

Удаляются признаки, отмеченные в исходном исследовании как дублирующие или неиспользуемые.

In [12]:
raw_train_df.drop(columns=["fact-region", "region", "registration-region", 'tp-foreign'], inplace=True)

In [13]:
num_columns = raw_train_df.select_dtypes(include=["int64", "float64"]).columns.to_list()

## 8. EDA числовых признаков

Функция строит histogram и boxplot, а выбросы определяются по правилу 1.5 × IQR.

# Числовые

In [14]:
def plot_numeric_analysis(df, col):
    """Строит гистограмму + boxplot для числового признака с аннотациями"""
    
    # Очистка от NaN
    data = df[col].dropna()
    n = len(data)
    
    if n == 0:
        print(f"{col}: нет данных")
        return
    
    # Основные статистики
    mean = data.mean()
    median = data.median()
    mode = data.mode().iloc[0] if not data.mode().empty else np.nan
    variance = data.var()
    std = data.std()
    q1 = data.quantile(0.25)
    q3 = data.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outliers = data[(data < lower_bound) | (data > upper_bound)]
    n_outliers = len(outliers)
    
    # Фигура
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'{col.upper()}\nN={n}, Outliers={n_outliers} ({n_outliers/n*100:.1f}%)', fontsize=14)
    
    # --- Гистограмма ---
    ax1 = axes[0]
    ax1.hist(data, bins=50, edgecolor='black', alpha=0.7, color='skyblue')
    
    # Линии среднего, медианы, моды
    ax1.axvline(mean, color='red', linewidth=2, label=f'Mean = {mean:.2f}')
    ax1.axvline(median, color='green', linewidth=2, linestyle='--', label=f'Median = {median:.2f}')
    if not np.isnan(mode):
        ax1.axvline(mode, color='orange', linewidth=2, linestyle=':', label=f'Mode = {mode:.2f}')
    
    ax1.set_xlabel(col)
    ax1.set_ylabel('Frequency')
    ax1.legend()
    ax1.grid(alpha=0.3)
    
    # --- Boxplot ---
    ax2 = axes[1]
    bp = ax2.boxplot(data, orientation='vertical', patch_artist=True,
                      boxprops=dict(facecolor='lightblue'),
                      whiskerprops=dict(linewidth=2),
                      capprops=dict(linewidth=2),
                      medianprops=dict(color='red', linewidth=2),
                      flierprops=dict(marker='o', markerfacecolor='red', markersize=5, alpha=0.5))
    
    # Добавляем статистику в виде текста
    stats_text = (
        f'Mean: {mean:.2f}\n'
        f'Median: {median:.2f}\n'
        f'Mode: {mode:.2f}\n'
        f'Std: {std:.2f}\n'
        f'Variance: {variance:.2f}\n'
        f'Q1: {q1:.2f}\n'
        f'Q3: {q3:.2f}\n'
        f'IQR: {iqr:.2f}\n'
        f'Min: {data.min():.2f}\n'
        f'Max: {data.max():.2f}\n'
        f'Outliers: {n_outliers}'
    )
    ax2.text(1.1, 0.95, stats_text, transform=ax2.transAxes, 
             fontsize=8, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    ax2.set_title(f'Boxplot with Outliers (n={n_outliers})')
    ax2.set_ylabel(col)
    ax2.set_xticklabels([''])
    
    plt.tight_layout()
    plt.show()
    
    # Дополнительно печатаем для отчёта
    print(f"\n=== {col} ===")
    print(f"Mean: {mean:.2f}")
    print(f"Median: {median:.2f}")
    print(f"Mode: {mode:.2f}")
    print(f"Std: {std:.2f}")
    print(f"Variance: {variance:.2f}")
    print(f"Q1: {q1:.2f}, Q3: {q3:.2f}, IQR: {iqr:.2f}")
    print(f"Min: {data.min():.2f}, Max: {data.max():.2f}")
    print(f"Outliers: {n_outliers} ({n_outliers/n*100:.1f}%)")
    print("-"*50)


# for col in num_columns:
#     plot_numeric_analysis(raw_train_df, col)

## 9. Финальный список удаляемых признаков

Список сгруппирован по исходным причинам удаления: дубли, константы и редкие признаки. Значения списка не изменены.

In [15]:
drop_cols = [
    # 1. Дублирующиеся региональные признаки
    
    # 2. Дублирующиеся равенства адресов (оставляем только reg-fact-and-post-equality)
    'reg-and-fact-equality',
    'post-and-fact-equality',
    'reg-and-post-equality',
    'reg-fact-post-and-last-credit-equality',
    
    # 3. Дублирующиеся просрочки (оставляем только max-delinquency-amount)
    'total-of-delinquencies',
    'max-delinquency-no',
    'mean-delinquency-amount',
    
    # 4. Монополисты / бесполезные
    'driving-license',         # все значения = 0
    
    # 5. Слишком редкие (< 5%)
    'cottage',                 # 1%
    'garage',                  # 2% (добавил, т.к. тоже редкий)
    'land',                    # 4%
    'reg-phone',               # 6%
    
]

In [16]:
raw_train_df.drop(columns=drop_cols, inplace=True)

## 10. Импутация для baseline-модели

Числовые признаки заполняются медианой, категориальные — первой модой. Этот блок повторяет исходную baseline-логику; production-вариант без leakage реализован в [`src/train.py`](../src/train.py).

In [17]:
numeric_cols = raw_train_df.select_dtypes(include=["int64", "float64"]).columns
raw_train_df[numeric_cols] = raw_train_df[numeric_cols].fillna(raw_train_df[numeric_cols].median())

In [18]:
# Заполнение категориальных признаков модой
cat_cols = raw_train_df.select_dtypes(include=["object", "category"]).columns

for col in cat_cols:
    mode_val = raw_train_df[col].mode()[0]  # берем первую моду
    raw_train_df[col] = raw_train_df[col].fillna(mode_val)

In [19]:
# Контроль состояния перед обучением
assert raw_train_df.isna().sum().sum() == 0, "После импутации остались пропуски"
cat_cols = raw_train_df.select_dtypes(include=["object", "category"]).columns
print(f"Признаков после отбора: {raw_train_df.shape[1] - 1}")
print(f"Категориальных признаков для CatBoost: {len(cat_cols)}")

Признаков после отбора: 33
Категориальных признаков для CatBoost: 9


## 11. Разбиение данных

Формируются train, validation и test-части для baseline-сравнения моделей.

In [20]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(raw_train_df.drop(columns=["target"]), raw_train_df["target"])
X_test, X_val, y_test, y_val = train_test_split(X_test, y_test)

## 12. Baseline CatBoost

Сравниваются модель с параметрами по умолчанию и модель с балансировкой классов.

In [21]:

model_default = cb.CatBoostClassifier(
    early_stopping_rounds=50,         # Остановка через 50 шагов без улучшения
    verbose=10,
    eval_metric='AUC',
)
model_default.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    cat_features=cat_cols.tolist()
)

Learning rate set to 0.065429
0:	test: 0.6315862	best: 0.6315862 (0)	total: 73.7ms	remaining: 1m 13s
10:	test: 0.6612842	best: 0.6674377 (8)	total: 243ms	remaining: 21.8s
20:	test: 0.6741112	best: 0.6750185 (16)	total: 410ms	remaining: 19.1s
30:	test: 0.6801955	best: 0.6806719 (28)	total: 588ms	remaining: 18.4s
40:	test: 0.6943353	best: 0.6943353 (40)	total: 761ms	remaining: 17.8s
50:	test: 0.7014853	best: 0.7014853 (50)	total: 923ms	remaining: 17.2s
60:	test: 0.7030273	best: 0.7030273 (60)	total: 1.1s	remaining: 17s
70:	test: 0.7057221	best: 0.7057221 (70)	total: 1.28s	remaining: 16.7s
80:	test: 0.7042601	best: 0.7057876 (71)	total: 1.45s	remaining: 16.4s
90:	test: 0.7062131	best: 0.7068750 (88)	total: 1.64s	remaining: 16.4s
100:	test: 0.7065513	best: 0.7070277 (99)	total: 1.81s	remaining: 16.1s
110:	test: 0.7078133	best: 0.7078133 (110)	total: 1.97s	remaining: 15.8s
120:	test: 0.7088352	best: 0.7088970 (114)	total: 2.15s	remaining: 15.6s
130:	test: 0.7081115	best: 0.7090861 (125)	tot

CatBoostClassifier(early_stopping_rounds=50, eval_metric='AUC', verbose=10)

In [22]:

model_weighted = cb.CatBoostClassifier(
    iterations=1500,
    learning_rate=0.005,
    depth=4,                  # Уменьшим глубину, чтобы не ловить шум
    l2_leaf_reg=10,           # Усилим регуляризацию
    early_stopping_rounds=100,
    eval_metric='AUC',
    auto_class_weights='Balanced',
    verbose=100,
)
model_weighted.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    cat_features=cat_cols.tolist()
)

0:	test: 0.6454151	best: 0.6454151 (0)	total: 12.1ms	remaining: 18.2s
100:	test: 0.6859707	best: 0.6859707 (100)	total: 1.23s	remaining: 17.1s
200:	test: 0.6859926	best: 0.6872509 (173)	total: 2.43s	remaining: 15.7s
300:	test: 0.6909059	best: 0.6911277 (294)	total: 3.77s	remaining: 15s
400:	test: 0.6944117	best: 0.6946117 (391)	total: 4.96s	remaining: 13.6s
500:	test: 0.6982303	best: 0.6982303 (498)	total: 6.26s	remaining: 12.5s
600:	test: 0.7011252	best: 0.7011252 (600)	total: 7.66s	remaining: 11.5s
700:	test: 0.7026963	best: 0.7027072 (697)	total: 9.05s	remaining: 10.3s
800:	test: 0.7038564	best: 0.7039183 (797)	total: 10.4s	remaining: 9.06s
900:	test: 0.7052784	best: 0.7053439 (896)	total: 11.7s	remaining: 7.75s
1000:	test: 0.7063004	best: 0.7063004 (1000)	total: 13s	remaining: 6.48s
1100:	test: 0.7073732	best: 0.7073732 (1100)	total: 14.4s	remaining: 5.2s
1200:	test: 0.7083479	best: 0.7083515 (1198)	total: 15.7s	remaining: 3.92s
1300:	test: 0.7088679	best: 0.7088679 (1300)	total: 1

CatBoostClassifier(auto_class_weights='Balanced', depth=4, early_stopping_rounds=100, eval_metric='AUC', iterations=1500, l2_leaf_reg=10, learning_rate=0.005, verbose=100)

## 13. Сравнение качества

Выводится classification report для обеих baseline-моделей.

In [23]:
from sklearn.metrics import classification_report

# Без весов
y_pred_default = model_default.predict(X_val)
print("=== БЕЗ ВЕСОВ ===")
print(classification_report(y_val, y_pred_default))

# С весами
y_pred_weighted = model_weighted.predict(X_val)
print("=== С ВЕСАМИ ===")
print(classification_report(y_val, y_pred_weighted))

=== БЕЗ ВЕСОВ ===
              precision    recall  f1-score   support

           0       0.87      1.00      0.93      1368
           1       0.20      0.00      0.01       201

    accuracy                           0.87      1569
   macro avg       0.54      0.50      0.47      1569
weighted avg       0.79      0.87      0.81      1569

=== С ВЕСАМИ ===
              precision    recall  f1-score   support

           0       0.93      0.64      0.76      1368
           1       0.21      0.66      0.32       201

    accuracy                           0.65      1569
   macro avg       0.57      0.65      0.54      1569
weighted avg       0.84      0.65      0.70      1569



## 14. Анализ порога классификации

Для weighted-модели сравниваются пороги 0.5, 0.6, 0.7 и 0.8.

In [24]:
probs = model_weighted.predict_proba(X_val)[:, 1]

# Попробуйте повысить порог, чтобы отсечь ложные срабатывания
for threshold in [0.5, 0.6, 0.7, 0.8]:
    preds = (probs > threshold).astype(int)
    print(f"\nПорог {threshold}:")
    print(classification_report(y_val, preds))


Порог 0.5:
              precision    recall  f1-score   support

           0       0.93      0.64      0.76      1368
           1       0.21      0.66      0.32       201

    accuracy                           0.65      1569
   macro avg       0.57      0.65      0.54      1569
weighted avg       0.84      0.65      0.70      1569


Порог 0.6:
              precision    recall  f1-score   support

           0       0.90      0.88      0.89      1368
           1       0.27      0.30      0.28       201

    accuracy                           0.81      1569
   macro avg       0.58      0.59      0.59      1569
weighted avg       0.82      0.81      0.81      1569


Порог 0.7:
              precision    recall  f1-score   support

           0       0.88      0.98      0.93      1368
           1       0.37      0.07      0.12       201

    accuracy                           0.87      1569
   macro avg       0.62      0.53      0.52      1569
weighted avg       0.81      0.87     

## 15. Итоги и переход к production

Результаты EDA используются для формирования baseline и гипотез. Финальное обучение и серия MLflow-экспериментов выполняются через [`src/train.py`](../src/train.py) и [`src/run_experiments.py`](../src/run_experiments.py).